# 02 -- Models

Model/pipeline definitions: RFM + K-Means, the CNN2D architecture, the domain-general aspect-extraction function, and the fake-review ensemble scoring function.

In [ ]:
import os
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "app").is_dir() and (candidate / "data").is_dir():
            return candidate
        alt = candidate / "Olist_Marketplace_Platform"
        if (alt / "backend" / "app").is_dir() and (alt / "data").is_dir():
            return alt
    raise RuntimeError("Could not locate the project root above this notebook.")


PROJECT_ROOT = _find_project_root(Path.cwd())
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
print("Project root:", PROJECT_ROOT)


In [ ]:
# Standalone setup: RFM needs a merged orders dataframe (§3.8 in the original
# notebook); reusing the canonical pre-built one instead of re-deriving the merge
# here keeps this file focused on the model itself. CNN2D's default constructor
# args reference these constants, defined in the original notebook's §6B.1.
import pandas as pd
import plotly.express as px

from app.ml.utils import get_device

device = get_device()
df = pd.read_parquet("data/processed/orders_enriched.parquet")

CNN_MAX_WORDS = 30_000
CNN_MAX_LEN = 100
CNN_EMBEDDING_DIM = 100


### 5.6 RFM Customer Segmentation

**R**ecency (days since last order), **F**requency (number of orders), **M**onetary
(total spend) — computed per customer and clustered with K-Means into business-friendly
segments. (Note: this notebook's companion pipeline notebook does the same analysis at the
**seller** level; this is the **customer**-level view.)

In [ ]:
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
import numpy as np
import os
import pickle

snapshot_date = df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = df.groupby("customer_unique_id").agg(
    Recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),
    Frequency=("order_id", "nunique"),
    Monetary=("total_payment_value", "sum"),
).reset_index()

# FIX: this cell previously fit StandardScaler directly on the raw RFM values (no
# log1p step). Frequency/Monetary are heavily right-skewed (typical e-commerce
# spend/order-count distributions); K-Means uses Euclidean distance, so untransformed
# outliers pull cluster centers toward them, producing one giant "everyone average"
# cluster plus a tiny "high spender" outlier cluster instead of behaviourally
# distinct segments. Verified on this dataset: pre-fix, 94% of customers landed in
# two clusters that were statistically indistinguishable on Frequency and Monetary,
# differing only on Recency.
#
# log1p + scale are now ONE fitted Pipeline object (not two separate fit/transform
# calls), and persisted, so the exact same transform is guaranteed to apply again at
# serving time instead of a separately-reconstructed "log then scale" step that isn't
# guaranteed to match.
rfm_pipeline = Pipeline([
    ("log", FunctionTransformer(np.log1p, inverse_func=np.expm1, validate=True)),
    ("scale", StandardScaler()),
])
rfm_scaled = rfm_pipeline.fit_transform(rfm[["Recency", "Frequency", "Monetary"]])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
rfm["Cluster"] = kmeans.fit_predict(rfm_scaled)

cluster_summary = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean().round(1)
cluster_summary["Customer_Count"] = rfm["Cluster"].value_counts()
print(cluster_summary.sort_values("Monetary", ascending=False))

# Persist the fitted pipeline + kmeans model so serving code can apply the identical
# transform used here (see the train/serve-skew fix note above).
os.makedirs("artifacts", exist_ok=True)
with open("artifacts/rfm_scaler.pkl", "wb") as f:
    pickle.dump(rfm_pipeline, f)
with open("artifacts/rfm_kmeans.pkl", "wb") as f:
    pickle.dump(kmeans, f)
print("Saved: artifacts/rfm_scaler.pkl, artifacts/rfm_kmeans.pkl")


In [48]:
# Map clusters to business-friendly labels based on the summary above
# (ranking by Monetary desc: highest spenders -> Champion, ... -> At Risk)
ranked_clusters = cluster_summary.sort_values("Monetary", ascending=False).index.tolist()
labels_by_rank = ["Champion", "Loyal Customer", "Potential Loyal", "At Risk"]
cluster_label_map = {cluster: labels_by_rank[i] for i, cluster in enumerate(ranked_clusters)}
rfm["Segment"] = rfm["Cluster"].map(cluster_label_map)

segment_counts = rfm["Segment"].value_counts()

fig = px.bar(
    segment_counts, x=segment_counts.index, y=segment_counts.values,
    title="Customer Segments (RFM + K-Means)",
    labels={"x": "Segment", "y": "Number of Customers"},
    text_auto=True, color=segment_counts.index,
)
fig.show()


**📌 Insight:** Given that most customers are one-time buyers (§5.1), the "Champion"
and "Loyal Customer" segments here are necessarily small — they represent the rare repeat
buyers worth protecting with retention campaigns, while the much larger "At Risk" /
"Potential Loyal" segments are the actual target for converting one-time buyers into
repeat customers.

### 6B.2 Build the CNN2D Model

In [56]:
import torch.nn as nn
import torch.nn.functional as F


class CNN2DReviewSentiment(nn.Module):
    '''Multi-branch n-gram CNN2D: Embedding -> reshape to 2D -> parallel Conv2D branches
    (one per n-gram filter size) -> BatchNorm2d -> global max-pool -> concat -> Dense.'''

    def __init__(self, vocab_size=CNN_MAX_WORDS, max_len=CNN_MAX_LEN, embed_dim=CNN_EMBEDDING_DIM,
                 num_filters=32, filter_sizes=(2, 3, 4, 5), dropout_rate=0.5):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim, padding_idx=0)
        # Element-wise dropout right after embedding — PyTorch stand-in for Keras SpatialDropout1D(0.2)
        self.embedding_dropout = nn.Dropout(0.2)

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels=1, out_channels=num_filters, kernel_size=(fs, embed_dim)),
                nn.BatchNorm2d(num_filters),
                nn.ReLU(),
            )
            for fs in filter_sizes
        ])

        self.dropout1 = nn.Dropout(dropout_rate)
        self.dense1 = nn.Linear(num_filters * len(filter_sizes), 32)
        self.dropout2 = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(32, 1)  # raw logit; apply sigmoid at inference time

    def forward(self, x):
        # x: (batch, max_len) integer token ids
        x = self.embedding(x)                                # (batch, max_len, embed_dim)
        x = self.embedding_dropout(x)
        x = x.unsqueeze(1)                                    # (batch, 1, max_len, embed_dim)

        branch_outputs = []
        for branch in self.branches:
            b = branch(x)                                     # (batch, num_filters, H', 1)
            b = F.adaptive_max_pool2d(b, output_size=1)        # global max-pool -> (batch, num_filters, 1, 1)
            branch_outputs.append(b.flatten(1))                # (batch, num_filters)

        merged = torch.cat(branch_outputs, dim=1)
        out = self.dropout1(merged)
        out = F.relu(self.dense1(out))
        out = self.dropout2(out)
        return self.output_layer(out).squeeze(1)               # (batch,) raw logits


model_cnn2d = CNN2DReviewSentiment().to(device)
print(model_cnn2d)
total_params = sum(p.numel() for p in model_cnn2d.parameters() if p.requires_grad)
print(f"\nTrainable parameters: {total_params:,}")

CNN2DReviewSentiment(
  (embedding): Embedding(30000, 100, padding_idx=0)
  (embedding_dropout): Dropout(p=0.2, inplace=False)
  (branches): ModuleList(
    (0): Sequential(
      (0): Conv2d(1, 32, kernel_size=(2, 100), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (1): Sequential(
      (0): Conv2d(1, 32, kernel_size=(3, 100), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (2): Sequential(
      (0): Conv2d(1, 32, kernel_size=(4, 100), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
    (3): Sequential(
      (0): Conv2d(1, 32, kernel_size=(5, 100), stride=(1, 1))
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
    )
  )
  (dropout1): Dropout(p=0.5, inplace=False)
  (dens

## 7. Aspect-based sentiment (domain-general extraction)

**The problem this replaced**: the first ABSA implementation forced a sentiment score for
all 5 fixed aspects on *every* review regardless of whether the aspect was mentioned at
all ("product quality: Positive 75%" on a review that only discusses delivery) — a
hallucination bug.

**The fix**: gate each aspect behind an actual keyword/mention check (`app/ml/absa.py`).

**Generalizing it further**: the keyword lists were hand-authored for Olist's e-commerce
domain specifically — porting to a new domain (restaurants, hotels) meant rewriting them
by hand. `app/ml/aspect_extraction.py` replaces this with a **domain-general** pipeline
using RAKE (Rapid Automatic Keyword Extraction) to pull salient phrases directly from each
review's own text, then match them against an aspect category by stemmed word overlap —
no training data, no per-domain keyword authoring required.


In [ ]:
# app/ml/aspect_extraction.py -- the core idea (simplified)
def aspect_mentioned(text, aspect_category, extra_seeds=None):
    candidates = extract_candidate_terms(text, max_terms=15)   # RAKE, pure statistics, no model
    category_words = {_stem(w) for w in aspect_category.split()} | set(extra_seeds or [])
    return any(_stem(w) in category_words for phrase in candidates for w in phrase.split())

# Empirically validated on BOTH e-commerce and non-e-commerce (restaurant, hotel) example
# reviews (scripts/aspect_extraction_demo.py). Honest, measured finding: extraction itself
# generalizes with no retraining; a semantic-similarity layer was ALSO tried to catch
# aspects discussed without their own name (e.g. "flimsy" for "product quality") but
# measured unreliable in testing -- documented as unused rather than shipped anyway.


In [ ]:
# app/ml/fake_review_detection.py (core logic, simplified)
def score(self, text):
    tfidf_prob = self.tfidf_clf.predict_proba(self.vectorizer.transform([text]))[0, 1]
    if self.bert_model is None:            # TF-IDF-only mode (RAM-constrained hosts)
        return tfidf_prob
    bert_prob = softmax(self.bert_model(**tokenize(text)).logits)[0, 1]
    return (bert_prob + tfidf_prob) / 2.0   # ensemble

def _verdict(fake_probability, margin=0.1):
    if fake_probability >= 0.5 + margin: return "FAKE"
    if fake_probability <= 0.5 - margin: return "REAL"
    return "UNCERTAIN"    # <- honestly reports ambiguity instead of forcing a guess
